In [1]:
# Cell 1: Import all required libraries
import polars as pl
import pandas as pd
import numpy as np
from kloppy import impect
import requests
import io
from kloppy.utils import github_resolve_raw_data_url
from tqdm.notebook import tqdm  # Progress bars
import pickle
from pathlib import Path

print(" All imports successful!")

 All imports successful!


In [2]:
# Cell 2: Setup project paths
import os
from pathlib import Path

# 1. Start from where the notebook is
current_path = Path.cwd()

# 2. Find the Project Root (SoccerImpectHackathon)
# This looks for the folder that actually CONTAINS the 'data' folder
if (current_path / "data").exists():
    BASE_DIR = current_path
else:
    # If not here, check one level up (common if notebooks are in a subfolder)
    BASE_DIR = current_path.parent

# 3. Define absolute paths
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

# Ensure they exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root identified as: {BASE_DIR}")
print(f"Saving/Loading from: {PROCESSED_DIR}")

Project Root identified as: /Users/tanishbhilare/Desktop/SoccerImpectHackathon
Saving/Loading from: /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/processed


### Load Match and Squad Metadata

Retrieve match fixtures and team information from IMPECT Open Data repository. This includes 306 Bundesliga matches from the 2023/24 season, with home/away squad IDs needed for team-aware zone classification in later steps.

In [3]:
# Cell 3: Load match and squad metadata
print("Loading match metadata...")

match_url = github_resolve_raw_data_url(
    repository="ImpectAPI/open-data",
    branch="main",
    file="data/matches/matches_743.json"
)
squads_url = github_resolve_raw_data_url(
    repository="ImpectAPI/open-data",
    branch="main",
    file="data/squads/squads_743.json"
)

# Load matches
response = requests.get(match_url)
matches = (
    pl.read_json(io.StringIO(response.text))
    .unnest("matchDay")
    .rename({'iterationId': 'competitionId', 'id': 'matchId'})
    .drop(['idMappings', 'lastCalculationDate', 'name', 'available'])
    .with_columns([
        (pl.col("index") + 1).alias("matchDay")
    ])
    .drop("index")
)

# Load squads
response = requests.get(squads_url)
squads = (
    pl.read_json(io.StringIO(response.text))
    .drop(['type', 'gender', 'imageUrl', 'idMappings', 'access', 'countryId'])
)

# Join to get team names
matches = (
    matches
    .join(
        squads.rename({"name": "homeTeam"}),
        left_on="homeSquadId",
        right_on="id",
        how="left"
    )
    .join(
        squads.rename({"name": "awayTeam"}),
        left_on="awaySquadId",
        right_on="id",
        how="left"
    )
    .select(['competitionId', 'matchId', 'homeSquadId', 'awaySquadId', 
             'homeTeam', 'awayTeam', 'matchDay', 'scheduledDate'])
)

print(f" Loaded {len(matches)} matches")
print(f"\nFirst 5 matches:")
print(matches.head())

Loading match metadata...
 Loaded 306 matches

First 5 matches:
shape: (5, 8)
┌────────────┬─────────┬────────────┬────────────┬────────────┬────────────┬──────────┬────────────┐
│ competitio ┆ matchId ┆ homeSquadI ┆ awaySquadI ┆ homeTeam   ┆ awayTeam   ┆ matchDay ┆ scheduledD │
│ nId        ┆ ---     ┆ d          ┆ d          ┆ ---        ┆ ---        ┆ ---      ┆ ate        │
│ ---        ┆ i64     ┆ ---        ┆ ---        ┆ str        ┆ str        ┆ i64      ┆ ---        │
│ i64        ┆         ┆ i64        ┆ i64        ┆            ┆            ┆          ┆ str        │
╞════════════╪═════════╪════════════╪════════════╪════════════╪════════════╪══════════╪════════════╡
│ 743        ┆ 122838  ┆ 38         ┆ 33         ┆ SV Werder  ┆ FC Bayern  ┆ 1        ┆ 2023-08-18 │
│            ┆         ┆            ┆            ┆ Bremen     ┆ München    ┆          ┆ T18:30:00Z │
│ 743        ┆ 122839  ┆ 41         ┆ 37         ┆ Bayer 04   ┆ RasenBalls ┆ 1        ┆ 2023-08-19 │
│            

### Single Match Inspection

Load one match to inspect data structure, available columns, and event types before processing the full dataset. This ensures the data schema is understood and helps identify any potential issues early.

In [4]:
# Cell 4: Load ONE match to inspect structure
print("Loading first match to inspect structure...")

match_id = matches['matchId'][0]
home_team = matches.filter(pl.col('matchId') == match_id)['homeTeam'][0]
away_team = matches.filter(pl.col('matchId') == match_id)['awayTeam'][0]

print(f"\nMatch: {home_team} vs {away_team}")
print(f"Match ID: {match_id}")

# Load match data
dataset = impect.load_open_data(match_id=match_id, competition_id=743)

# Transform to standard coordinate system
df_sample = (
    dataset
    .transform(to_coordinate_system="secondspectrum")
    .to_df(engine="polars")
)

print(f"\n Total events: {len(df_sample)}")
print(f"\nEvent type distribution:")
print(df_sample.group_by('event_type').agg(pl.count().alias('count')).sort('count', descending=True))

print(f"\nAvailable columns ({len(df_sample.columns)}):")
for i, col in enumerate(df_sample.columns, 1):
    print(f"  {i}. {col}")

print(f"\nSample of first 3 events:")
print(df_sample.select([
    'event_id', 'event_type', 'team_id', 'player_id', 
    'coordinates_x', 'coordinates_y', 'result', 'success', 'period_id', 'timestamp'
]).head(3))

Loading first match to inspect structure...

Match: SV Werder Bremen vs FC Bayern München
Match ID: 122838


/Users/tanishbhilare/anaconda3/envs/soccer-hackathon/lib/python3.11/site-packages/kloppy/_providers/impect.py:88: UserWarning: 

You are about to use IMPECT public data.
By using this data, you are agreeing to the user agreement. 
The user agreement can be found here: https://github.com/ImpectAPI/open-data/blob/main/LICENSE.pdf

  warnings.warn(



 Total events: 3057

Event type distribution:
shape: (17, 2)
┌───────────────────────┬───────┐
│ event_type            ┆ count │
│ ---                   ┆ ---   │
│ str                   ┆ u32   │
╞═══════════════════════╪═══════╡
│ PASS                  ┆ 1000  │
│ GENERIC:RECEPTION     ┆ 831   │
│ CARRY                 ┆ 735   │
│ RECOVERY              ┆ 111   │
│ DUEL                  ┆ 106   │
│ …                     ┆ …     │
│ SUBSTITUTION          ┆ 10    │
│ GENERIC:NO_VIDEO      ┆ 10    │
│ GENERIC:OFFSIDE       ┆ 4     │
│ GENERIC:GOAL          ┆ 4     │
│ GENERIC:FINAL_WHISTLE ┆ 2     │
└───────────────────────┴───────┘

Available columns (22):
  1. event_id
  2. event_type
  3. period_id
  4. timestamp
  5. end_timestamp
  6. ball_state
  7. ball_owning_team
  8. team_id
  9. player_id
  10. coordinates_x
  11. coordinates_y
  12. end_coordinates_x
  13. end_coordinates_y
  14. receiver_player_id
  15. body_part_type
  16. set_piece_type
  17. result
  18. success
  19. du

/var/folders/tm/bnb1d19x3s99b2mrbgj6wt500000gn/T/ipykernel_54718/169270963.py:23: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  print(df_sample.group_by('event_type').agg(pl.count().alias('count')).sort('count', descending=True))


### Team-Aware Zone Classification

**Key Insight**: With `STATIC_HOME_AWAY` orientation:
- Home team attacks → (defends at x<-17.5, attacks at x>17.5)
- Away team attacks ← (defends at x>17.5, attacks at x<-17.5)

**Critical Fix**: We pass `team_id` and `home_squad_id` to flip zone logic for away teams.

**Impact**: Ensures defensive/attacking actions are correctly attributed regardless of which team performed them.


In [5]:
# Cell 5: Define pitch zone classification
def classify_zone(x_coordinate, team_id, home_squad_id):
    """
    TEAM-AWARE zone classification
    """
    if x_coordinate is None or team_id is None:
        return None
    
    is_home_team = (str(team_id) == str(home_squad_id))
    
    if is_home_team:
        if x_coordinate < -17.5:
            return "defensive_third"
        elif x_coordinate > 17.5:
            return "attacking_third"
        else:
            return "middle_third"
    else:
        # AWAY TEAM - FLIPPED!
        if x_coordinate > 17.5:
            return "defensive_third"
        elif x_coordinate < -17.5:
            return "attacking_third"
        else:
            return "middle_third"

### Match Loading Function - Team-Aware Zone Classification

Defines function to load individual matches and apply team-aware zone classification, accounting for the fact that home teams attack right while away teams attack left in the STATIC_HOME_AWAY coordinate system used by IMPECT data.

In [6]:
# Cell 6: Function to load and process a single match with team-aware zone classification
def load_and_process_match(match_id, home_squad_id, away_squad_id, competition_id=743):
    """
    Load a single match and add team-aware zone classification.
    
    With STATIC_HOME_AWAY orientation:
    - Home team always attacks right (toward +52.5)
    - Away team always attacks left (toward -52.5)
    
    Args:
        match_id: int, match identifier
        home_squad_id: str, home team squad ID
        away_squad_id: str, away team squad ID
        competition_id: int, competition identifier (default 743 for Bundesliga)
        
    Returns:
        polars.DataFrame: Processed match events with team-aware zones
    """
    # Load match data from IMPECT API
    dataset = impect.load_open_data(match_id=match_id, competition_id=competition_id)
    
    # Transform coordinates only (orientation is already STATIC_HOME_AWAY in IMPECT data)
    df = (
        dataset
        .transform(to_coordinate_system="secondspectrum")
        .to_df(engine="polars")
    )
    
    # Add match metadata columns
    df = df.with_columns([
        pl.lit(match_id).alias('match_id'),
        pl.lit(home_squad_id).alias('home_squad_id'),
        pl.lit(away_squad_id).alias('away_squad_id')
    ])
    
    # Apply team-aware zone classification for event start position
    zones = []
    for row in df.iter_rows(named=True):
        if row['coordinates_x'] is not None and row['team_id'] is not None:
            zone = classify_zone(
                row['coordinates_x'], 
                row['team_id'], 
                home_squad_id
            )
            zones.append(zone)
        else:
            zones.append(None)
    
    df = df.with_columns([
        pl.Series('zone', zones)
    ])
    
    # Apply team-aware zone classification for event end position (for passes and carries)
    end_zones = []
    for row in df.iter_rows(named=True):
        if row['end_coordinates_x'] is not None and row['team_id'] is not None:
            zone = classify_zone(
                row['end_coordinates_x'], 
                row['team_id'], 
                home_squad_id
            )
            end_zones.append(zone)
        else:
            end_zones.append(None)
    
    df = df.with_columns([
        pl.Series('end_zone', end_zones)
    ])
    
    return df


# Test function on first match
print("Testing match loading with team-aware zones")

# Get first match information
first_match_id = matches['matchId'][0]
match_info = matches.filter(pl.col('matchId') == first_match_id).row(0, named=True)

print(f"\nMatch: {match_info['homeTeam']} vs {match_info['awayTeam']}")
print(f"Match ID: {first_match_id}")
print(f"Home Squad ID: {match_info['homeSquadId']}")
print(f"Away Squad ID: {match_info['awaySquadId']}")

# Load match with team-aware zones
test_df = load_and_process_match(
    match_id=first_match_id,
    home_squad_id=str(match_info['homeSquadId']),
    away_squad_id=str(match_info['awaySquadId'])
)

print(f"\nProcessed {len(test_df)} events")

# Display zone distribution
print(f"\nZone distribution:")
print(test_df.group_by('zone').agg(pl.len().alias('count')).sort('zone'))

# Display sample events
print(f"\nSample events with zones:")
print(test_df.select([
    'event_type', 'team_id', 'coordinates_x', 'zone', 
    'end_coordinates_x', 'end_zone'
]).head(5))


# Verify zones are correctly assigned
print("Zone verification")

# Split events by team
home_events = test_df.filter(pl.col('team_id') == str(match_info['homeSquadId']))
away_events = test_df.filter(pl.col('team_id') == str(match_info['awaySquadId']))

# Calculate average X positions
home_avg_x = home_events['coordinates_x'].mean()
away_avg_x = away_events['coordinates_x'].mean()

print(f"\n{match_info['homeTeam']} (Home team):")
print(f"  Average X position: {home_avg_x:.2f}")
print(f"  Expected: Negative (they start on left, attack right)")
print(f"  Zone distribution:")
print(home_events.group_by('zone').agg(pl.len().alias('count')).sort('zone'))

print(f"\n{match_info['awayTeam']} (Away team):")
print(f"  Average X position: {away_avg_x:.2f}")
print(f"  Expected: Positive (they start on right, attack left)")
print(f"  Zone distribution:")
print(away_events.group_by('zone').agg(pl.len().alias('count')).sort('zone'))

# Final validation check
print("\nValidation check:")
if home_avg_x < 0 and away_avg_x > 0:
    print("  PASS: Home team has negative avg X")
    print("  PASS: Away team has positive avg X")
    print("  SUCCESS: Zones are correctly classified")
else:
    print(f"  FAIL: Home avg X = {home_avg_x:.2f} (expected negative)")
    print(f"  FAIL: Away avg X = {away_avg_x:.2f} (expected positive)")
    print("  ERROR: Zones are still incorrect - check code")


Testing match loading with team-aware zones

Match: SV Werder Bremen vs FC Bayern München
Match ID: 122838
Home Squad ID: 38
Away Squad ID: 33

Processed 3057 events

Zone distribution:
shape: (4, 2)
┌─────────────────┬───────┐
│ zone            ┆ count │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ null            ┆ 57    │
│ attacking_third ┆ 606   │
│ defensive_third ┆ 977   │
│ middle_third    ┆ 1417  │
└─────────────────┴───────┘

Sample events with zones:
shape: (5, 6)
┌──────────────────┬─────────┬───────────────┬─────────────────┬─────────────────┬─────────────────┐
│ event_type       ┆ team_id ┆ coordinates_x ┆ zone            ┆ end_coordinates ┆ end_zone        │
│ ---              ┆ ---     ┆ ---           ┆ ---             ┆ _x              ┆ ---             │
│ str              ┆ str     ┆ f64           ┆ str             ┆ ---             ┆ str             │
│                  ┆         ┆               ┆                 ┆ f64       

### Full Season Data Loading with Zone Classification

Iterate through all 306 Bundesliga matches, applying team-aware zone classification to each event based on home/away team context, then combine and cache the complete dataset for subsequent analysis steps.

In [7]:
# Cell 7: Load all matches with team-aware zone classification
import time

# Check for existing cache
cache_file = PROCESSED_DIR / "all_matches_with_zones.parquet"

# Delete old cache if it exists (to rebuild with fixed zones)
if cache_file.exists():
    print(f"Found existing cache at {cache_file}")
    print("Deleting old cache to rebuild with team-aware zones...")
    cache_file.unlink()
    print("Old cache deleted.")

print(f"\nLoading all {len(matches)} matches with team-aware zones...")

all_events_list = []
failed_matches = []

start_time = time.time()

# Loop through all matches
for idx, match_id in enumerate(matches['matchId'].to_list(), 1):
    try:
        # Get match information for home/away squad IDs
        match_info = matches.filter(pl.col('matchId') == match_id).row(0, named=True)
        home_squad_id = str(match_info['homeSquadId'])
        away_squad_id = str(match_info['awaySquadId'])
        
        # Load and process match with team-aware zones
        df = load_and_process_match(
            match_id=match_id,
            home_squad_id=home_squad_id,
            away_squad_id=away_squad_id
        )
        
        all_events_list.append(df)
        
        # Print progress every 50 matches
        if idx % 50 == 0:
            elapsed = time.time() - start_time
            avg_time = elapsed / idx
            remaining = (len(matches) - idx) * avg_time
            print(f"  Processed {idx}/{len(matches)} matches | "
                  f"Elapsed: {elapsed/60:.1f}min | "
                  f"Remaining: {remaining/60:.1f}min")
            
    except Exception as e:
        print(f"  Failed to load match {match_id}: {e}")
        failed_matches.append(match_id)
        continue

# Combine all matches into single dataframe
print("\nCombining all matches...")
all_events = pl.concat(all_events_list, how="diagonal")

# Save to cache
print(f"Saving to cache: {cache_file}...")
all_events.write_parquet(cache_file)

# Report timing
elapsed_total = time.time() - start_time
print(f"\nLoaded {len(all_events):,} events from {len(all_events_list)} matches")
print(f"Total time: {elapsed_total/60:.1f} minutes")

if failed_matches:
    print(f"\nWarning: {len(failed_matches)} matches failed to load")
    print(f"Failed match IDs: {failed_matches}")

# Display summary statistics
print("DATASET SUMMARY")

print(f"Total events: {len(all_events):,}")
print(f"Total matches: {all_events['match_id'].n_unique()}")
print(f"Total players: {all_events['player_id'].n_unique()}")

print(f"\nEvent type distribution (top 10):")
print(all_events.group_by('event_type').agg(pl.len().alias('count')).sort('count', descending=True).head(10))

print(f"\nZone distribution:")
print(all_events.group_by('zone').agg(pl.len().alias('count')).sort('zone'))


# Verify zones are correctly assigned across all matches
print("ZONE VERIFICATION (Sample Match)")

# Take a sample match to verify
sample_match_id = matches['matchId'][0]
sample_match = all_events.filter(pl.col('match_id') == sample_match_id)
sample_info = matches.filter(pl.col('matchId') == sample_match_id).row(0, named=True)

# Split by team
home_team_events = sample_match.filter(pl.col('team_id') == str(sample_info['homeSquadId']))
away_team_events = sample_match.filter(pl.col('team_id') == str(sample_info['awaySquadId']))

# Check average positions
home_avg_x = home_team_events['coordinates_x'].mean()
away_avg_x = away_team_events['coordinates_x'].mean()

print(f"\nSample: {sample_info['homeTeam']} vs {sample_info['awayTeam']}")
print(f"Home team ({sample_info['homeTeam']}) avg X: {home_avg_x:.2f}")
print(f"Away team ({sample_info['awayTeam']}) avg X: {away_avg_x:.2f}")

# Validation
if home_avg_x < 0 and away_avg_x > 0:
    print("\nValidation: PASS")
    print("  - Home team has negative avg X (attacks right)")
    print("  - Away team has positive avg X (attacks left)")
    print("  - Zone classification is correct")
else:
    print("\nValidation: FAIL")
    print(f"  - Home team avg X: {home_avg_x:.2f} (expected: negative)")
    print(f"  - Away team avg X: {away_avg_x:.2f} (expected: positive)")
    print("  - ERROR: Check zone classification logic")


Found existing cache at /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/processed/all_matches_with_zones.parquet
Deleting old cache to rebuild with team-aware zones...
Old cache deleted.

Loading all 306 matches with team-aware zones...
  Processed 50/306 matches | Elapsed: 0.3min | Remaining: 1.8min
  Processed 100/306 matches | Elapsed: 0.7min | Remaining: 1.4min
  Processed 150/306 matches | Elapsed: 1.0min | Remaining: 1.0min
  Processed 200/306 matches | Elapsed: 1.3min | Remaining: 0.7min
  Processed 250/306 matches | Elapsed: 1.6min | Remaining: 0.4min
  Processed 300/306 matches | Elapsed: 1.9min | Remaining: 0.0min

Combining all matches...
Saving to cache: /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/processed/all_matches_with_zones.parquet...

Loaded 962,990 events from 306 matches
Total time: 1.9 minutes
DATASET SUMMARY
Total events: 962,990
Total matches: 306
Total players: 494

Event type distribution (top 10):
shape: (10, 2)
┌───────────────────┬──────

### Data Quality and Integrity Checks

Validate dataset quality through systematic checks of null values, coordinate ranges, event distributions and zone classifications. Confirm data meets expected parameters and team-aware zones are correctly assigned before downstream analysis.

In [8]:
# Cell 8: Data quality checks
print("DATA QUALITY CHECKS")

# Check for null values in key columns
print("\n1. Null values in key columns:")
key_cols = ['event_type', 'team_id', 'player_id', 'coordinates_x', 'coordinates_y', 'zone']
for col in key_cols:
    null_count = all_events[col].null_count()
    null_pct = (null_count / len(all_events)) * 100
    print(f"  {col:20s}: {null_count:>8,} nulls ({null_pct:>5.2f}%)")

# Check coordinate ranges
print("\n2. Coordinate ranges:")
x_min = all_events['coordinates_x'].min()
x_max = all_events['coordinates_x'].max()
y_min = all_events['coordinates_y'].min()
y_max = all_events['coordinates_y'].max()

print(f"  X coordinates: [{x_min:.2f}, {x_max:.2f}]")
print(f"  Y coordinates: [{y_min:.2f}, {y_max:.2f}]")
print(f"  Expected X: [-52.5, 52.5]")
print(f"  Expected Y: [-34.0, 34.0]")

# Validate coordinate ranges
if -52.5 <= x_min <= -52.0 and 52.0 <= x_max <= 52.5:
    print("  Validation: X coordinates within expected range")
else:
    print(f"  Warning: X coordinates outside expected range")

if -34.5 <= y_min <= -33.5 and 33.5 <= y_max <= 34.5:
    print("  Validation: Y coordinates within expected range")
else:
    print(f"  Warning: Y coordinates outside expected range")

# Check events per match
print("\n3. Events per match:")
events_per_match = all_events.group_by('match_id').agg(pl.len().alias('event_count'))
print(f"  Min events per match: {events_per_match['event_count'].min()}")
print(f"  Max events per match: {events_per_match['event_count'].max()}")
print(f"  Mean events per match: {events_per_match['event_count'].mean():.0f}")
print(f"  Std dev: {events_per_match['event_count'].std():.0f}")

# Check for reasonable distribution
avg_events = events_per_match['event_count'].mean()
if 2500 <= avg_events <= 3500:
    print("  Validation: Events per match in reasonable range")
else:
    print(f"  Warning: Average events per match seems unusual")

# Sample events to inspect
print("\n4. Sample of 5 random events:")
print(all_events.sample(5).select([
    'event_type', 'team_id', 'player_id', 'zone', 'result', 'success'
]))

# Check zone distribution makes sense
print("\n5. Zone distribution analysis:")
zone_counts = all_events.group_by('zone').agg(pl.len().alias('count')).sort('zone')
print(zone_counts)

# Calculate zone percentages (FIX: use .item() to get scalar value)
total_with_zones = len(all_events) - all_events['zone'].null_count()
attacking_count = all_events.filter(pl.col('zone') == 'attacking_third').height  # Use .height instead of .count()
defensive_count = all_events.filter(pl.col('zone') == 'defensive_third').height
middle_count = all_events.filter(pl.col('zone') == 'middle_third').height

attacking_pct = (attacking_count / total_with_zones) * 100
defensive_pct = (defensive_count / total_with_zones) * 100
middle_pct = (middle_count / total_with_zones) * 100

print(f"\nZone percentages (excluding nulls):")
print(f"  Defensive third: {defensive_pct:.1f}%")
print(f"  Middle third: {middle_pct:.1f}%")
print(f"  Attacking third: {attacking_pct:.1f}%")

# Sanity check: middle third should have most events
if middle_pct > defensive_pct and middle_pct > attacking_pct:
    print("  Validation: Middle third has most events (expected)")
else:
    print("  Warning: Zone distribution unexpected - check classification")

# Check team balance
print("\n6. Team balance check:")
team_counts = all_events.group_by('team_id').agg(pl.len().alias('event_count')).sort('event_count', descending=True)
print(f"  Total teams: {len(team_counts)}")
print(f"  Events per team range: {team_counts['event_count'].min():,} to {team_counts['event_count'].max():,}")
print(f"  Mean events per team: {team_counts['event_count'].mean():,.0f}")

# Should have 18 teams (Bundesliga)
if len(team_counts) == 18:
    print("  Validation: Correct number of teams (18 for Bundesliga)")
else:
    print(f"  Warning: Expected 18 teams, found {len(team_counts)}")

print("Data quality checks complete")

DATA QUALITY CHECKS

1. Null values in key columns:
  event_type          :        0 nulls ( 0.00%)
  team_id             :    7,655 nulls ( 0.79%)
  player_id           :    7,655 nulls ( 0.79%)
  coordinates_x       :   19,281 nulls ( 2.00%)
  coordinates_y       :   19,281 nulls ( 2.00%)
  zone                :   19,281 nulls ( 2.00%)

2. Coordinate ranges:
  X coordinates: [-52.50, 52.50]
  Y coordinates: [-34.00, 34.00]
  Expected X: [-52.5, 52.5]
  Expected Y: [-34.0, 34.0]
  Validation: X coordinates within expected range
  Validation: Y coordinates within expected range

3. Events per match:
  Min events per match: 2314
  Max events per match: 4131
  Mean events per match: 3147
  Std dev: 314
  Validation: Events per match in reasonable range

4. Sample of 5 random events:
shape: (5, 6)
┌───────────────────┬─────────┬───────────┬─────────────────┬──────────┬─────────┐
│ event_type        ┆ team_id ┆ player_id ┆ zone            ┆ result   ┆ success │
│ ---               ┆ ---   

### Data Export and Pipeline Completion

Save all processed datasets to disk including 962,990 events with corrected zones, match metadata, squad information, and player statistics, completing the data loading pipeline and preparing inputs for the event scoring phase.

In [9]:
# Cell 9: Save metadata for downstream analysis
print("Saving metadata for later analysis...")

# Save matches dataframe
matches.write_parquet(PROCESSED_DIR / "matches_metadata.parquet")
print(f"Saved matches metadata: {len(matches)} matches")

# Save squads dataframe  
squads.write_parquet(PROCESSED_DIR / "squads_metadata.parquet")
print(f"Saved squads metadata: {len(squads)} teams")

# Create player metadata (needed for normalization and filtering)
player_metadata = all_events.group_by(['player_id', 'team_id']).agg([
    pl.len().alias('total_events'),
    pl.col('match_id').n_unique().alias('matches_played')
]).sort('total_events', descending=True)

player_metadata.write_parquet(PROCESSED_DIR / "player_metadata.parquet")
print(f"Saved player metadata: {len(player_metadata)} players")


print("STEP 1 COMPLETE - Data Loading")

print("\nFiles created:")
print(f"  1. {PROCESSED_DIR / 'all_matches_with_zones.parquet'}")
print(f"  2. {PROCESSED_DIR / 'matches_metadata.parquet'}")
print(f"  3. {PROCESSED_DIR / 'squads_metadata.parquet'}")
print(f"  4. {PROCESSED_DIR / 'player_metadata.parquet'}")

print("\nDataset statistics:")
print(f"  Total events: {len(all_events):,}")
print(f"  Total matches: {len(matches)}")
print(f"  Total players: {len(player_metadata)}")
print(f"  Total teams: {len(squads)}")


Saving metadata for later analysis...
Saved matches metadata: 306 matches
Saved squads metadata: 18 teams
Saved player metadata: 507 players
STEP 1 COMPLETE - Data Loading

Files created:
  1. /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/processed/all_matches_with_zones.parquet
  2. /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/processed/matches_metadata.parquet
  3. /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/processed/squads_metadata.parquet
  4. /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/processed/player_metadata.parquet

Dataset statistics:
  Total events: 962,990
  Total matches: 306
  Total players: 507
  Total teams: 18
